A timed, PDF-rendered AP Computer Science A Free-Response practice tool with four Java editors and AI grading via Gemini 2.5 Flash.

**Repos:** [pages](https://github.com/Open-Coding-Society/pages) (frontend) · [spring](https://github.com/Open-Coding-Society/spring) (backend)
**My PRs:** [#1268](https://github.com/Open-Coding-Society/pages/pull/1268), [#728](https://github.com/Open-Coding-Society/pages/pull/728), [#156](https://github.com/Open-Coding-Society/spring/pull/156), [#121](https://github.com/Open-Coding-Society/spring/pull/121), [#74](https://github.com/Open-Coding-Society/spring/pull/74)

## The problem

AP CSA students at our school had no realistic way to practice FRQs. PDFs from the College Board, random text editors, no timer, no feedback for days. I built one tool that fixes all of it.

## What it does

- Renders the real exam PDF on the left.
- Gives you four Java editors on the right with no "Run" button — real exams don't let you compile.
- Times you, switches to warning mode at 15 minutes and danger mode at 5, auto-submits at zero.
- Sends your code to a Spring Boot endpoint that calls Gemini and returns a score plus feedback.

## Architecture

```
Browser (GitHub Pages)              Spring Boot (EC2 + Docker + nginx)
─────────────────────              ──────────────────────────────────
PDF.js renderer            ──►     POST /api/gemini-frq/grade
4 × Java editors                      │
Timer FSM                             ├──► GeminiFrqService → Gemini 2.5 Flash
"Grade with AI" button                └──► JPA: Person, Group (JSON-blob grades)
```

Frontend stays static (free hosting, fast CDN). All secrets and state live behind one Spring service.

## Data structures

**Frontend — exam state.** A small finite state machine instead of a tangle of booleans:

```js
const ExamState = Object.freeze({
  IDLE: 'idle', RUNNING: 'running',
  WARNING: 'warning', DANGER: 'danger',
  SUBMITTED: 'submitted',
});
```

**Frontend — answers.** A `Map<questionId, AnswerEntry>` so iteration order matches question order at submit time.

**Backend — grades.** Instead of a wide grades table with one column per assignment, I stored grades as a JSON blob on the `Person` row:

```java
@Entity
public class Person {
    @Id @GeneratedValue private Long id;

    @Column(columnDefinition = "TEXT")
    private String grades;   // {"frq-2024": 8.5, "frq-2023": 7.0}
}
```

Trade-off: less queryable, but no migration every time a new assignment shows up. For our scale that was the right call.

## Algorithms

**Group → Person grade propagation.** When a group gets a grade, every member's blob is updated. O(m · k) where m = group size, k = grades per person. Both are small (~30 and ~50), so it runs in a couple ms.

**Timer.** One `setInterval` per second, but each tick recomputes `endTime − Date.now()` instead of decrementing a counter. That way, if the tab loses focus, the timer can't drift. O(1) per tick.

## OOP and patterns

- **Encapsulation.** `GeminiFrqController` knows nothing about Gemini. It calls `service.grade(frq, code)` and gets back a `GradeResult`. The service hides the API key, request body, and parsing.
- **Polymorphism.** `Person` and `Group` both implement `Gradable`. The grading service takes a `Gradable` and doesn't care which one it got.
- **Dependency injection.** Every Spring component is constructor-injected.
- **State pattern.** The frontend FSM above.

## Code highlight — the security change (PR #156)

The grading endpoint was behind auth, which meant students had to log in to a teacher tool just to practice. One line in `SecurityConfig`:

```java
http.authorizeHttpRequests(auth -> auth
    .requestMatchers("/api/gemini-frq/grade").permitAll()
    .requestMatchers("/api/**").authenticated()
);
```

In review, the instructor asked "did you purposely leave security open to the general public on POST?" Yes — but next iteration adds rate-limiting per IP so it can't be abused as a free Gemini relay.

## Gist export — how it ties the systems together

The Gist exporter (PR #728) started as a small utility but ended up being the connective tissue between every part of the OCS Pages platform that students write code in. The idea: any page with code editors can opt in with one front-matter line, and students get a "share my work" button for free.

```
                  ┌──────────────────────────────────────┐
                  │  Any page with `show_gist_export:    │
                  │  true` in its front matter           │
                  └──────────────────────────────────────┘
                                   │
        ┌──────────────────────────┼──────────────────────────┐
        ▼                          ▼                          ▼
  Lesson pages              FRQ Simulator              Tangibles / Hacks
  (CodeRunner               (4 Java editors            (multi-snippet
   homework problems)        per exam)                  practice pages)
        │                          │                          │
        └──────────────────────────┼──────────────────────────┘
                                   ▼
                  ┌──────────────────────────────────────┐
                  │  exportToGist()                      │
                  │  • Walks every .coderunner on page   │
                  │  • Reads question name + code        │
                  │  • Bundles into { files: {...} }     │
                  │  • POSTs to GitHub Gist API          │
                  │  • Copies link to clipboard          │
                  └──────────────────────────────────────┘
                                   │
                                   ▼
                  ┌──────────────────────────────────────┐
                  │  GitHub Gist (one URL, all answers)  │
                  └──────────────────────────────────────┘
                                   │
        ┌──────────────────────────┼──────────────────────────┐
        ▼                          ▼                          ▼
  Teacher review            Submission tracker          Student portfolio
  (open link, grade         (Eshika's feature —          (paste link into
   code in one place)        auto-pulls Gist URL)        any LMS / form)
```

**The workflow in plain English.** A student finishes a lesson or an FRQ exam. They click "Export to Gist." The function walks every code editor on the page, pulls each editor's question label and code, bundles them into a single Gist request, and posts to the GitHub API with a deployment-time key. The returned URL is auto-copied to their clipboard. From there:

- For **lessons**, the student pastes the link into their homework submission.
- For the **FRQ Simulator**, the link is the artifact a teacher reviews alongside the AI-generated score from `/api/gemini-frq/grade`. Score from Gemini, code from Gist — that's the full grading picture.
- For **Eshika's submission tracker** (next-sprint integration), the tracker reads the Gist URL directly from the click, no copy-paste step needed.

**Why a Gist and not our own backend?** Three reasons:

1. **Free hosting for code snippets.** GitHub already does this well; we'd be reinventing the wheel.
2. **Versioning.** Every export is a revision in the Gist's history, so a student can prove what they had at a given moment.
3. **No auth complexity on the student side.** The Gist key is held by the deployed site, not asked from each student. One key, many users.

**The front-matter opt-in.** Pages choose whether to expose the button:

```yaml
---
layout: post
show_gist_export: true
---
```

If the flag isn't set, the button never renders. That keeps the feature opt-in per page rather than globally on, which matters because not every page has code editors.

## My PRs

| PR | Repo | What I owned |
|----|------|--------------|
| [#74](https://github.com/Open-Coding-Society/spring/pull/74) | spring | Gemini placeholder + JSON-blob grade refactor on Person/Group |
| [#121](https://github.com/Open-Coding-Society/spring/pull/121) | spring | Built `POST /api/gemini-frq/grade` with Gemini prompt + Postman tests |
| [#156](https://github.com/Open-Coding-Society/spring/pull/156) | spring | `SecurityConfig` change — permitAll on grading endpoint |
| [#728](https://github.com/Open-Coding-Society/pages/pull/728) | pages | "Code → Gist" exporter with front-matter opt-in |
| [#1268](https://github.com/Open-Coding-Society/pages/pull/1268) | pages | The full simulator: PDF rendering, 4 editors, timer FSM, AI grading |

All five merged.

## Testing

`GeminiFrqController` is tested with `@WebMvcTest` and `MockMvc`, with the service mocked so tests don't hit the real Gemini API:

```java
@Test
void gradeReturns200AndScore() throws Exception {
    when(service.grade(anyString(), anyString()))
        .thenReturn(new GradeResult(8.5, "Good logic."));

    mvc.perform(post("/api/gemini-frq/grade")
            .contentType(MediaType.APPLICATION_JSON)
            .content("{\"frqText\":\"...\",\"studentCode\":\"...\"}"))
       .andExpect(status().isOk())
       .andExpect(jsonPath("$.score").value(8.5));
}
```

A Postman collection in `docs/postman/` covers happy path, missing-field 400, and oversized-body cases. JaCoCo coverage on the controller is 100% line / 100% branch.
